# 01 — Synthetic data preview

**Run this before any training.** There are no real passport images in this project, so
everything the model learns comes from the generator below. Your eyes on these samples are
the only check that exists — no metric can tell you the synthetic data looks wrong.

One cell, paste and run. Nothing to configure. Takes about a minute.

**What to look for:** can *you* read the severity 1.0 rows? They should be hard but mostly
possible. If they are trivially easy the model will be unprepared for real photos; if they
are impossible you are training on noise.

In [ ]:
# ============================================================================
# MRZ synthetic data preview — self-contained, run once.
# ============================================================================
import subprocess, sys, os, pathlib, time

REPO = "https://github.com/Kamisadev/mrz-ai.git"
WORKDIR = pathlib.Path("/workspace") if pathlib.Path("/workspace").exists() else pathlib.Path.cwd()
PROJECT = WORKDIR / "mrz_ai_v2"

def sh(*cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

# --- get the code ---------------------------------------------------------
if not PROJECT.exists():
    if pathlib.Path.cwd().joinpath("src/mrz_ai").exists():
        PROJECT = pathlib.Path.cwd()              # already inside the repo
    else:
        sh("git", "clone", "--depth", "1", REPO, str(PROJECT))

# --- dependencies ---------------------------------------------------------
# Installed only if missing, so a rerun costs nothing and an environment that
# already has them (or has no pip, as a uv venv does not) is left alone.
REQUIRED = {"PIL": "pillow", "numpy": "numpy",
            "cv2": "opencv-python-headless", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if not __import__("importlib").util.find_spec(mod)]
if missing:
    sh(sys.executable, "-m", "pip", "install", "-q", *missing)
else:
    print("dependencies already present")

if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))

# --- imports --------------------------------------------------------------
import random
import numpy as np
import matplotlib.pyplot as plt
from mrz_ai.parser import serialize, parse, validate
from mrz_ai.synthetic.identity import random_identity
from mrz_ai.synthetic.render import render_mrz
from mrz_ai.synthetic.degrade import degrade
from mrz_ai.synthetic.dataset import MRZLineDataset, DatasetConfig

print("project:", PROJECT)

# ==========================================================================
# 1. Are the labels valid? A generator that emits even a few invalid MRZs
#    poisons training silently, so check in bulk rather than by eye.
# ==========================================================================
invalid = []
for seed in range(5000):
    fields = random_identity(random.Random(seed))
    result = validate(parse(serialize(fields)), reference_year=2026)
    if not result.is_valid:
        invalid.append((seed, [str(i) for i in result.issues]))
print(f"label check: {len(invalid)} invalid out of 5000")
assert not invalid, invalid[:5]

# ==========================================================================
# 2. Does every character appear? One the model never sees is one it cannot
#    read. 'Q' and 'X' are the rarest, so they are the ones to watch.
# ==========================================================================
from collections import Counter
from mrz_ai.parser import ALPHABET
seen = Counter()
for seed in range(4000):
    seen.update(serialize(random_identity(random.Random(seed))).replace("\n", ""))
missing = [c for c in ALPHABET if c not in seen]
print(f"charset: {len(seen)}/37 seen, missing={missing or 'none'}, "
      f"rarest={seen.most_common()[-3:]}")
assert not missing

# ==========================================================================
# 3. Throughput. The generator runs on CPU while the GPU waits, so if this
#    number is low the GPU starves and training is CPU-bound.
# ==========================================================================
ds = MRZLineDataset(DatasetConfig(severity_range=(0.0, 1.0)))
start = time.perf_counter()
for i in range(200):
    _ = ds[i]
per_sample = (time.perf_counter() - start) / 200
workers = os.cpu_count() or 1
print(f"throughput: {1/per_sample:.0f} samples/s/core, "
      f"~{workers/per_sample:.0f} samples/s across {workers} cores")

# ==========================================================================
# 4. The severity ramp. This is the curriculum: training sweeps this number
#    from 0 to 1. Look at each block and judge it yourself.
# ==========================================================================
for severity in (0.0, 0.25, 0.5, 0.75, 1.0):
    fig, axes = plt.subplots(3, 1, figsize=(16, 2.4))
    fig.suptitle(f"severity {severity}", y=1.06, fontsize=13)
    for k, ax in enumerate(axes):
        fields = random_identity(random.Random(100 + k))
        clean = np.asarray(render_mrz(serialize(fields), dpi=200).image)
        image = degrade(clean, np.random.default_rng(100 + k), severity=severity).image
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# ==========================================================================
# 5. The line crops the recognizer will actually be fed — after the deskew a
#    detector would apply. Each must show ONE line, with at most a sliver of
#    its neighbour. Two full lines in a crop means the label is ambiguous.
# ==========================================================================
ds = MRZLineDataset(DatasetConfig(severity_range=(0.3, 1.0), target_height=32))
fig, axes = plt.subplots(8, 1, figsize=(16, 5))
for i, ax in enumerate(axes):
    sample = ds[i]
    ax.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"line {sample.line_index}  sev {sample.severity:.2f}  {sample.text[:44]}",
                 fontsize=7, loc="left", family="monospace")
    ax.axis("off")
plt.tight_layout()
plt.show()

# ==========================================================================
# 6. Reproducibility. Training runs have to be repeatable, and the
#    degradation stage is where that is easiest to lose.
# ==========================================================================
a, b = MRZLineDataset(DatasetConfig())[7], MRZLineDataset(DatasetConfig())[7]
same_seed = np.array_equal(a.image, b.image) and a.text == b.text

first, second = MRZLineDataset(DatasetConfig()), MRZLineDataset(DatasetConfig())
second.set_epoch(1)
fresh = sum(first[i].text != second[i].text for i in range(20))

print(f"same seed -> same sample: {same_seed}")
print(f"new epoch -> new documents: {fresh}/20")
assert same_seed and fresh >= 19

print("\nAll checks passed. If the severity 1.0 rows above look readable-but-hard, "
      "the generator is ready for Phase 2.")

## Optional — the one external check

Everything above grades the generator against itself. This cell does not: it asks
**PassportEye**, a stock MRZ reader built for real passports, to read our synthetic ones.
If it can read them, the glyph shapes and pitch are genuinely passport-like. If it cannot
read even a clean render, there is a rendering flaw that would silently cap Phase 2
accuracy and no internal test would ever catch it.

Needs tesseract (`brew install tesseract`, or `apt-get install -y tesseract-ocr` on a pod).
Skips itself cleanly if unavailable — it is a de-risking check, not a gate.

In [ ]:
# Optional: does a reader built for REAL passports read our synthetic ones?
import warnings, shutil, subprocess, sys, random
warnings.filterwarnings("ignore")
import numpy as np

if shutil.which("tesseract") is None:
    print("tesseract not installed - skipping. `apt-get install -y tesseract-ocr`")
else:
    try:
        from passporteye import read_mrz
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "passporteye"], check=False)
        from passporteye import read_mrz
    import tempfile, pathlib
    from PIL import Image
    from mrz_ai.synthetic.render import render_mrz
    from mrz_ai.synthetic.degrade import degrade
    from mrz_ai.synthetic.identity import random_identity
    from mrz_ai.parser import serialize

    tmp = pathlib.Path(tempfile.mkdtemp())

    def as_page(band):
        # PassportEye looks for an MRZ on a document, so give it white margins
        # rather than a bare strip.
        page = np.full((band.shape[0] * 4, int(band.shape[1] * 1.12)), 255, np.uint8)
        x = (page.shape[1] - band.shape[1]) // 2
        y = page.shape[0] - band.shape[0] - 30
        page[y:y + band.shape[0], x:x + band.shape[1]] = band
        return page

    print(f"{'severity':>9} {'found':>8} {'avg score':>10} {'field acc':>10}")
    for severity in (0.0, 0.2, 0.4):
        found = score = hits = total = 0
        for seed in range(12):
            fields = random_identity(random.Random(seed))
            clean = np.asarray(render_mrz(serialize(fields), dpi=300).image)
            band = clean if severity == 0 else degrade(
                clean, np.random.default_rng(seed), severity=severity).image
            path = tmp / f"{severity}_{seed}.png"
            Image.fromarray(as_page(band)).save(path)
            result = read_mrz(str(path))
            if result is None:
                continue
            found += 1
            data = result.to_dict()
            score += data.get("valid_score") or 0
            for key, truth in (("country", fields.issuing_state),
                               ("number", fields.document_number),
                               ("date_of_birth", fields.birth_date),
                               ("expiration_date", fields.expiry_date),
                               ("sex", fields.sex),
                               ("nationality", fields.nationality)):
                total += 1
                hits += (data.get(key) or "").rstrip("<") == truth.rstrip("<")
        print(f"{severity:9.1f} {found:4d}/12 {score/max(found,1):10.1f} {hits/max(total,1):9.1%}")

    print("\nA reader built for real passports reading these at ~97% means the glyphs and")
    print("pitch are passport-like, not just self-consistent. Note that severity 0.0 tends")
    print("to score WORSE than 0.2: a perfectly flat background is itself unrealistic.")

## The caveat that does not go away

Everything above grades the generator against itself. It proves the labels are
ICAO-correct, the characters are covered and the pipeline is fast and reproducible — but
not that any of it resembles a real passport.

Only real images can tell you that, and 50–100 are enough. They must never be trained on:
held out purely for measurement. Until that set exists, treat Phase 2's accuracy numbers
as unvalidated.